# 03 - Clima (INMET, estação A312)

**Objetivo:** montar o clima semanal de Teresina-PI e juntar com os casos.

O clima do InfoDengue só começa em junho de 2023 e não tem chuva. O INMET cobre o período inteiro. A estação automática do INMET em Teresina é a A312.

Fonte: [Portal INMET – dados históricos](https://portal.inmet.gov.br/dadoshistoricos) (ZIPs baixados em 26/10/2026)

## 1. Extrair os CSVs da A312

Os ZIPs anuais do INMET têm ~500 estações e ficam fora do Git. Aqui copio só o arquivo da A312 de cada ano, sem alterar nada, pra `data/raw/inmet/a312`.

In [1]:
import zipfile
from pathlib import Path

# ZIPs anuais do INMET
PASTA_INMET = Path("../data/raw/inmet")

# Pasta onde ficam só os CSVs da estação A312
PASTA_A312 = PASTA_INMET / "a312"

# Cria a pasta se ainda não existir
PASTA_A312.mkdir(parents=True, exist_ok=True)

# Lista os ZIPs em ordem de ano
zips = sorted(PASTA_INMET.glob("*.zip"))
[caminho.name for caminho in zips]

['2015.zip',
 '2016.zip',
 '2017.zip',
 '2018.zip',
 '2019.zip',
 '2020.zip',
 '2021.zip',
 '2022.zip',
 '2023.zip',
 '2024.zip',
 '2025.zip',
 '2026.zip']

In [ ]:
# Copia o CSV da A312 de cada ZIP, byte por byte
for caminho_zip in zips:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        for nome in arquivo_zip.namelist():
            # Procura pelo nome e não pelo caminho: em alguns anos o CSV fica numa pasta dentro do ZIP
            if "_A312_" in nome.upper():
                destino = PASTA_A312 / Path(nome).name

                # Só extrai se ainda não existir (mesmo padrão do 01_coleta: não sobrescreve o raw)
                if destino.exists():
                    print("já existe:", destino.name)
                else:
                    destino.write_bytes(arquivo_zip.read(nome))
                    print("extraído:", destino.name)

extraído: INMET_NE_PI_A312_TERESINA_01-01-2015_A_31-12-2015.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2016_A_31-12-2016.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2017_A_31-12-2017.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2018_A_31-12-2018.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2019_A_31-12-2019.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2020_A_31-12-2020.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2021_A_31-12-2021.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2022_A_31-12-2022.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2023_A_31-12-2023.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2024_A_31-12-2024.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2025_A_31-12-2025.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2026_A_31-08-2026.CSV
extraído: INMET_NE_PI_A312_TERESINA_01-01-2026_A_30-09-2026.CSV


In [4]:
# Confere quantos arquivos saíram e o tamanho total
csvs_a312 = sorted(PASTA_A312.glob("*.CSV"))

tamanho_total = 0
for caminho in csvs_a312:
    tamanho_total = tamanho_total + caminho.stat().st_size

print("Arquivos:", len(csvs_a312))
print("Tamanho total:", round(tamanho_total / 1024 / 1024, 1), "MB")

Arquivos: 13
Tamanho total: 8.8 MB


In [5]:
# Confere se cada arquivo extraído é idêntico ao que está dentro do ZIP (prova de que é dado bruto mesmo)
identicos = 0
diferentes = 0

for caminho_zip in zips:
    with zipfile.ZipFile(caminho_zip) as arquivo_zip:
        for nome in arquivo_zip.namelist():
            if "_A312_" in nome.upper():
                extraido = PASTA_A312 / Path(nome).name
                if extraido.read_bytes() == arquivo_zip.read(nome):
                    identicos = identicos + 1
                else:
                    diferentes = diferentes + 1
                    print("DIFERENTE:", extraido.name)

print("Idênticos:", identicos, "| Diferentes:", diferentes)

Idênticos: 13 | Diferentes: 0


### Conclusões do passo 1

- 13 CSVs da A312 extraídos (8,8 MB), todos idênticos aos de dentro dos ZIPs.
- 2026 veio em dois arquivos que se sobrepõem (01/01 a 31/08 e 01/01 a 30/09): juntando os dois, janeiro a agosto de 2026 ficaria duplicado.
- O clima vai até 30/09/2026 e os casos até 11/10/2026: as últimas semanas (que já são abertas) ficam sem clima.